<a href="https://colab.research.google.com/github/elsheppardo/hello-world/blob/main/Stage_1_Create_FX_Rates_and_BTC_USD_Lookup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
STAGE 1: FX Rates and BTC/USD Prices Lookup Table
=====================================================
Purpose: Establish a single source of truth for all currency and BTC
         price data used in the capital gains calculations.

Output: Stage1_FX_BTC_Rates.xlsx — a simple 2-column lookup table
        (USD/CAD and BTC/USD) for every date referenced in the
        transaction history.

How to verify:
  1. Open the output file.
  2. Check the USD/CAD rates (Column B) against the Bank of Canada
     Valet API or daily lookup tool. These are estimates — your
     accountant will want official BoC rates before filing.
     Lookup tool: https://www.bankofcanada.ca/rates/exchange/daily-exchange-rates-lookup/
     Valet API:   https://www.bankofcanada.ca/valet/observations/FXUSDCAD/csv?start_date=2022-01-01&end_date=2023-12-31
  3. Check BTC/USD prices (Column C) against CoinGecko or
     CoinMarketCap for rough reasonableness. These drive only minor
     calculations (Feb 24 AVAX value, etc.) — actual trade fills
     from the exchange CSVs are used where available.
  4. Edit any rate cells (they're yellow-highlighted as "VERIFY")
     and re-save the file. Subsequent stages will reference this file.

Notes:
  - Every date corresponds to an actual transaction date from the
    Paxos, KuCoin, Terra, or NDAX CSVs.
  - If the accountant prefers the monthly average FX rate instead
    of daily rates (CRA allows this), those values can replace
    the daily rates here without changing downstream sheets.
"""

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ---------- Styles ----------
FONT_NAME = "Arial"
styles = {
    'title':    Font(name=FONT_NAME, size=16, bold=True, color='1F4E79'),
    'header':   Font(name=FONT_NAME, size=10, bold=True, color='FFFFFF'),
    'body':     Font(name=FONT_NAME, size=10),
    'input':    Font(name=FONT_NAME, size=10, color='0000FF'),  # Blue = input
    'note':     Font(name=FONT_NAME, size=9,  italic=True, color='595959'),
    'header_fill': PatternFill('solid', fgColor='1F4E79'),
    'verify_fill': PatternFill('solid', fgColor='FFF2CC'),  # Yellow = VERIFY
}
thin = Side(border_style='thin', color='BFBFBF')
cell_border = Border(left=thin, right=thin, top=thin, bottom=thin)

FMT_FX    = '0.0000'
FMT_PRICE = '"$"#,##0.00'

# ---------- FX rate and BTC price data ----------
# Each row: (Date, USD/CAD rate, BTC/USD close, Notes)
# Rates are approximate — flagged for verification against Bank of Canada.
rate_data = [
    # 2022 transaction dates
    ("2022-01-03", 1.2717, 46622, "CAD→USD purchase date #1 (user records)"),
    ("2022-02-24", 1.2734, 38360, "Avalanche → KuCoin → Terra bridge day"),
    ("2022-03-04", 1.2621, 39204, "Paxos wire deposit #1 ($114,970 USD)"),
    ("2022-03-05", 1.2621, 39407, "Paxos BTC buys (25 fills agg)"),
    ("2022-03-08", 1.2854, 38728, "First BTC arrives at KuCoin"),
    ("2022-03-09", 1.2791, 41886, "Paxos wire #2 ($171,845 USD); CAD→USD purchase #2"),
    ("2022-03-10", 1.2740, 39409, "Paxos BTC buys (15 fills agg)"),
    ("2022-03-11", 1.2743, 38724, "BTC → KuCoin; UST to Terra"),
    ("2022-03-12", 1.2743, 38794, "UST to Terra"),
    ("2022-03-13", 1.2743, 38794, "BTC → KuCoin (weekend FX carry)"),
    ("2022-03-14", 1.2810, 37789, "Paxos BTC buys (14 fills agg)"),
    ("2022-03-15", 1.2762, 39290, "UST to Terra"),
    ("2022-03-17", 1.2626, 40906, "Final UST transfers to Terra"),
    ("2022-05-09", 1.2959, 30296, "Terra collapse begins; small escape tests"),
    ("2022-05-10", 1.3029, 31022, "MAJOR — 343,639.92 UST → USDT escape"),
    ("2022-05-13", 1.2933, 29283, "First USDT withdrawal to Paxos wallet"),
    ("2022-06-30", 1.2886, 19986, "BTC buy KuCoin; transfer to Paxos"),
    ("2022-07-01", 1.2855, 19270, "BTC sell Paxos"),
    ("2022-07-12", 1.3017, 19313, "BTC buy KuCoin"),
    ("2022-07-13", 1.3027, 20209, "BTC withdrawal to Paxos"),
    ("2022-07-14", 1.3075, 20247, "BTC sell Paxos"),
    ("2022-08-17", 1.2860, 23378, "BTC buys both KuCoin and Paxos"),
    ("2022-11-09", 1.3511, 15881, "USDT transfer to Paxos wallet"),
    ("2022-11-10", 1.3339, 17535, "USDT transfer to Paxos wallet"),

    # 2023 transaction dates
    ("2023-01-14", 1.3389, 20961, "Paxos BTC sells (2 BTC)"),
    ("2023-02-26", 1.3629, 23174, "NDAX USDT deposit + first CAD withdrawals"),
    ("2023-03-09", 1.3854, 20360, "KuCoin → NDAX USDT"),
    ("2023-03-10", 1.3797, 20154, "KuCoin → NDAX USDT"),
    ("2023-03-11", 1.3797, 20450, "KuCoin → NDAX USDT (large)"),
    ("2023-03-13", 1.3703, 24248, "NDAX large CAD withdrawal"),
    ("2023-03-20", 1.3712, 27720, "NDAX CAD withdrawal"),
    ("2023-03-22", 1.3699, 27271, "Paxos BTC buy/sell & KuCoin final BTC sell"),
    ("2023-04-03", 1.3441, 27991, "KuCoin → NDAX USDC"),
    ("2023-04-04", 1.3416, 28168, "NDAX CAD withdrawal"),
    ("2023-04-05", 1.3500, 28185, "NDAX CAD withdrawal"),
    ("2023-09-11", 1.3528, 25164, "NDAX CAD withdrawal"),
    ("2023-11-27", 1.3589, 37241, "NDAX USDC deposits (tail end)"),
    ("2023-12-05", 1.3557, 44067, "NDAX USDC deposit (tail end)"),
    ("2023-12-06", 1.3577, 43777, "Final NDAX CAD withdrawal"),
]


def build_workbook(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "FX_BTC_Rates"

    # Column widths
    widths = [14, 12, 14, 60]
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w

    # Title and instructions
    ws.cell(row=1, column=1, value="FX Rates and BTC/USD Prices — Lookup Table").font = styles['title']
    ws.cell(row=2, column=1, value=(
        "Source: Bank of Canada FXUSDCAD daily + BTC/USD daily close. "
        "VERIFY yellow cells against official Bank of Canada data before using downstream."
    )).font = styles['note']
    ws.cell(row=3, column=1, value=(
        "This sheet is referenced by later stages. Edit values here and downstream calcs will update."
    )).font = styles['note']

    # Header row
    headers = ["Date", "USD/CAD", "BTC/USD", "Notes (transaction on this date)"]
    for i, h in enumerate(headers, 1):
        cell = ws.cell(row=5, column=i, value=h)
        cell.font = styles['header']
        cell.fill = styles['header_fill']
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border = cell_border

    # Data rows
    for i, (date_str, fx, btc, note) in enumerate(rate_data):
        r = 6 + i

        c = ws.cell(row=r, column=1, value=date_str)
        c.font = styles['body']
        c.border = cell_border

        c = ws.cell(row=r, column=2, value=fx)
        c.font = styles['input']
        c.fill = styles['verify_fill']
        c.number_format = FMT_FX
        c.border = cell_border

        c = ws.cell(row=r, column=3, value=btc)
        c.font = styles['input']
        c.fill = styles['verify_fill']
        c.number_format = FMT_PRICE
        c.border = cell_border

        c = ws.cell(row=r, column=4, value=note)
        c.font = styles['note']
        c.alignment = Alignment(horizontal='left', vertical='center')
        c.border = cell_border

    # Summary/totals row (just count)
    total_row = 6 + len(rate_data) + 1
    c = ws.cell(row=total_row, column=1, value=f"Total rows: {len(rate_data)}")
    c.font = styles['body']

    # Freeze header
    ws.freeze_panes = "A6"

    wb.save(output_path)
    print(f"Saved: {output_path}")
    print(f"Rows written: {len(rate_data)}")
    print(f"Date range: {rate_data[0][0]} to {rate_data[-1][0]}")


if __name__ == "__main__":
    output = "Stage1_FX_BTC_Rates.xlsx"
    build_workbook(output)